# Stage 5 — Earth Network QA Gate

| Field | Value |
|-------|-------|
| **Pipeline stage** | Stage 5 — Final Earth network generation and QA |
| **Previous stage** | Stage 4 — Earth-Mars regime calibration (`notebooks/regime/00_calibration_overview.ipynb`, `docs/REGIME_SELECTION.md`) |
| **Next stage** | Stage 6 — Earth pair and label generation; Stage 7 — Raster patch construction; Stage 8 — Model training |
| **Purpose** | Verify that the regime-generated Earth networks are scientifically sound before committing to raster regeneration and model retraining. Flags outlier basins, checks label distributions, and issues an explicit PASS / FAIL decision. |
| **Inputs from previous stages** | `data/results/build_earth_features_reg{A,B,C}_stats.csv` (per-basin run stats from Stage 5 feature build); `data/results/master_dataset_reg{A,B,C}.csv` (labeled pair datasets from Stage 6) |
| **Outputs produced by this stage** | `data/results/stage5_earth_network_qa_report.csv` (per-basin QA table, written only on PASS); console PASS / FAIL decision with flag list |
| **Decision gate** | **Must PASS before regenerating rasters (Stage 7), retraining CNN or XGBoost (Stage 8), or running Mars inference (Stages 10–11).** A FAIL means at least one hard criterion is violated; review the flagged basins and either re-run `scripts/cli/build_earth_features_regime.py` with adjusted parameters or explicitly acknowledge the issue before proceeding. |

---

**Scientific context:** The three regimes (regA/B/C) use different area thresholds and Strahler pruning to bracket Earth network complexity calibrated against Mars drainage density (see `docs/REGIME_SELECTION.md`). This notebook checks whether the generated networks produce reasonable pair statistics — not whether the science is correct, but whether the data is coherent enough to train on.

## 0. Configuration

In [ ]:
# ── Gate thresholds ────────────────────────────────────────────────────
# Basins with fewer pairs than this are flagged (soft warn, not hard fail).
MIN_PAIRS_PER_BASIN = 10

# Each regime must have at least this many total pairs across all basins.
MIN_TOTAL_PAIRS = 5_000

# Basins whose touching ratio is outside [LO, HI] with >MIN_PAIRS_PER_BASIN
# pairs are flagged. Ratios near 0 or 1 suggest labeling problems.
TOUCHING_RATIO_LO = 0.05
TOUCHING_RATIO_HI = 0.95

# Write the QA report CSV only when the gate passes.
WRITE_REPORT = True

## 1. Imports and paths

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from channel_heads.io.paths import RESULTS_DIR, CROPPED_DEMS_DIR
from channel_heads.regimes import REGIMES, Regime
from channel_heads.basin_config import get_basin_config, LOCAL_TO_PAPER_BASIN

REGIMES_ORDER = ["regA", "regB", "regC"]
FEATURE_COLS = [
    "delta_L",
    "orientation_diff_deg",
    "headhead_dist_norm",
    "apex_angle_deg",
    "proximity_profile_norm",
]

print("RESULTS_DIR:", RESULTS_DIR)
print("Regimes:", REGIMES_ORDER)

## 2. Load regime datasets

In [ ]:
stats: dict[str, pd.DataFrame] = {}
master: dict[str, pd.DataFrame] = {}

for reg in REGIMES_ORDER:
    stats_path = RESULTS_DIR / f"build_earth_features_{reg}_stats.csv"
    master_path = RESULTS_DIR / f"master_dataset_{reg}.csv"

    if not stats_path.exists():
        raise FileNotFoundError(
            f"{stats_path} not found — run scripts/cli/build_earth_features_regime.py "
            f"--regime {reg} first."
        )
    if not master_path.exists():
        raise FileNotFoundError(
            f"{master_path} not found — run scripts/cli/build_earth_features_regime.py "
            f"--regime {reg} first."
        )

    stats[reg] = pd.read_csv(stats_path)
    master[reg] = pd.read_csv(master_path)
    s, m = stats[reg], master[reg]
    print(f"{reg}: {len(s)} basins, {s.n_pairs.sum():,} pairs "
          f"({s.n_touching.sum():,} touching), "
          f"{m['touching'].isna().sum()} NaN labels")

## 3. Cross-regime pair summary

In [ ]:
# Build a wide table: one row per basin, one column-group per regime.
frames = []
for reg in REGIMES_ORDER:
    df = stats[reg][['basin', 'n_pairs', 'n_touching', 'error']].copy()
    df['touching_ratio'] = df['n_touching'] / df['n_pairs'].clip(lower=1)
    df.columns = ['basin', f'{reg}_pairs', f'{reg}_touching', f'{reg}_error', f'{reg}_ratio']
    frames.append(df.set_index('basin'))

summary = frames[0].join(frames[1]).join(frames[2]).reset_index()
summary = summary.sort_values('basin')

# Add paper names for readability.
summary['paper_name'] = summary['basin'].map(
    lambda b: LOCAL_TO_PAPER_BASIN.get(b, b)
)

display_cols = ['basin', 'paper_name'] + [
    f'{r}_{x}' for r in REGIMES_ORDER for x in ['pairs', 'ratio']
]
pd.set_option('display.float_format', '{:.3f}'.format)
display(summary[display_cols].to_string(index=False))

## 4. Flag analysis

In [ ]:
flags: list[str] = []   # hard-fail conditions
warnings: list[str] = []  # soft flags

for reg in REGIMES_ORDER:
    s = stats[reg]
    regime = REGIMES[reg]

    # Hard: error basins
    err = s[s['error'].notna()]
    for _, row in err.iterrows():
        flags.append(f"{reg}/{row['basin']}: run error — {row['error']}")

    # Hard: total pairs below minimum
    total = s['n_pairs'].sum()
    if total < MIN_TOTAL_PAIRS:
        flags.append(f"{reg}: only {total:,} total pairs (need ≥{MIN_TOTAL_PAIRS:,})")

    # Soft: low pair count basins
    low = s[s['n_pairs'] < MIN_PAIRS_PER_BASIN]
    for _, row in low.iterrows():
        warnings.append(
            f"{reg}/{row['basin']}: {row['n_pairs']} pairs (< {MIN_PAIRS_PER_BASIN})"
        )

    # Soft: touching ratio extremes (only basins with enough pairs)
    sig = s[s['n_pairs'] >= MIN_PAIRS_PER_BASIN].copy()
    sig['ratio'] = sig['n_touching'] / sig['n_pairs']
    extreme = sig[(sig['ratio'] < TOUCHING_RATIO_LO) | (sig['ratio'] > TOUCHING_RATIO_HI)]
    for _, row in extreme.iterrows():
        warnings.append(
            f"{reg}/{row['basin']}: touching ratio {row['ratio']:.2f} "
            f"(outside [{TOUCHING_RATIO_LO}, {TOUCHING_RATIO_HI}])"
        )

    # Hard: NaN in key feature columns
    m = master[reg]
    for col in FEATURE_COLS:
        if col in m.columns:
            n_nan = m[col].isna().sum()
            if n_nan > 0:
                flags.append(f"{reg}: {n_nan} NaN values in feature column '{col}'")

print(f"Hard flags ({len(flags)}):")
for f in flags:
    print(f"  ✗ {f}")
print()
print(f"Soft warnings ({len(warnings)}):")
for w in warnings:
    print(f"  ⚠ {w}")

## 5. Feature distributions

In [ ]:
fig, axes = plt.subplots(
    len(FEATURE_COLS), len(REGIMES_ORDER),
    figsize=(4 * len(REGIMES_ORDER), 2.5 * len(FEATURE_COLS)),
    sharey='row',
)

COLORS = {True: '#d62728', False: '#1f77b4'}  # touching=red, non-touching=blue

for col_idx, reg in enumerate(REGIMES_ORDER):
    m = master[reg]
    for row_idx, feat in enumerate(FEATURE_COLS):
        ax = axes[row_idx, col_idx]
        if feat not in m.columns:
            ax.text(0.5, 0.5, f'{feat}\nnot found', ha='center', va='center',
                    transform=ax.transAxes, fontsize=9)
            continue
        for label, grp in m.groupby('touching'):
            vals = grp[feat].dropna()
            lo, hi = vals.quantile(0.01), vals.quantile(0.99)
            ax.hist(
                vals.clip(lo, hi), bins=40, alpha=0.55,
                color=COLORS[label],
                label='touching' if label else 'non-touching',
                density=True,
            )
        if row_idx == 0:
            ax.set_title(reg, fontsize=10, fontweight='bold')
        if col_idx == 0:
            ax.set_ylabel(feat, fontsize=8)
        ax.yaxis.set_major_locator(mticker.NullLocator())

axes[0, -1].legend(fontsize=7, loc='upper right')
fig.suptitle('Feature distributions by touching label and regime (1–99th pct clip)', y=1.01)
fig.tight_layout()
plt.show()

## 6. QC flag analysis

In [ ]:
for reg in REGIMES_ORDER:
    m = master[reg]
    if 'qc_flags' not in m.columns:
        print(f"{reg}: no qc_flags column")
        continue
    flagged = m[m['qc_flags'].notna() & (m['qc_flags'] != '')]
    total = len(m)
    pct = 100 * len(flagged) / total
    top = flagged['qc_flags'].value_counts().head(5)
    print(f"{reg}: {len(flagged):,}/{total:,} pairs have QC flags ({pct:.1f}%)")
    print(top.to_string())
    print()

## 7. Per-basin detail — flagged basins

Detailed breakdown for basins that appeared in the soft-warning list.

In [ ]:
# Extract basin names from warnings.
warned_basins: set[str] = set()
for w in warnings + flags:
    parts = w.split('/')
    if len(parts) >= 2:
        warned_basins.add(parts[1].split(':')[0].strip())

if not warned_basins:
    print("No flagged basins — all basins within expected ranges.")
else:
    print(f"Flagged basins: {sorted(warned_basins)}\n")
    for basin in sorted(warned_basins):
        try:
            cfg = get_basin_config(basin)
            print(f"{basin} ({cfg.get('full_name', basin)}):  "
                  f"lat={cfg.get('lat'):.1f}, area={cfg.get('area_km2'):.0f} km²")
        except Exception:
            print(f"{basin}: no basin config entry")

        for reg in REGIMES_ORDER:
            row = stats[reg][stats[reg]['basin'] == basin]
            if row.empty:
                print(f"  {reg}: not in stats")
                continue
            r = row.iloc[0]
            ratio = r['n_touching'] / max(r['n_pairs'], 1)
            print(f"  {reg}: {r['n_pairs']:4d} pairs, "
                  f"{r['n_touching']:4d} touching ({ratio:.2f})"
                  + (f", ERROR: {r['error']}" if pd.notna(r.get('error')) else ""))
        print()

## 8. DEM overview for low-pair basins (optional)

Loads the DEM thumbnail for each flagged basin to give visual context.
Requires `rasterio`. If not available, this cell is skipped.

In [ ]:
try:
    import rasterio
    from rasterio.enums import Resampling
    HAS_RASTERIO = True
except ImportError:
    HAS_RASTERIO = False
    print("rasterio not available — skipping DEM thumbnails.")

if HAS_RASTERIO and warned_basins:
    from channel_heads.training.regime import DEM_TO_BASIN
    basin_to_dem = {v: k for k, v in DEM_TO_BASIN.items()}

    plot_basins = sorted(warned_basins)[:4]  # at most 4
    fig, axes = plt.subplots(1, len(plot_basins),
                              figsize=(4 * len(plot_basins), 4))
    if len(plot_basins) == 1:
        axes = [axes]

    for ax, basin in zip(axes, plot_basins):
        dem_file = basin_to_dem.get(basin)
        dem_path = CROPPED_DEMS_DIR / dem_file if dem_file else None
        if dem_path is None or not dem_path.exists():
            ax.set_title(f"{basin}\n(DEM not found)")
            ax.axis('off')
            continue
        with rasterio.open(dem_path) as src:
            # Downsample to 256px on the longest side for quick display.
            scale = 256 / max(src.width, src.height)
            out_w = max(1, int(src.width * scale))
            out_h = max(1, int(src.height * scale))
            data = src.read(
                1, out_shape=(out_h, out_w),
                resampling=Resampling.bilinear
            ).astype(float)
        data[data == src.nodata if src.nodata is not None else False] = np.nan
        lo, hi = np.nanpercentile(data, [2, 98])
        ax.imshow(data, cmap='terrain', vmin=lo, vmax=hi, aspect='auto')
        ax.set_title(f"{basin}\n(DEM thumbnail)", fontsize=9)
        ax.axis('off')

    fig.suptitle('DEM thumbnails — flagged basins', fontsize=10)
    fig.tight_layout()
    plt.show()

## 9. Decision gate — PASS / FAIL

This cell is the formal Stage 5 gate. It raises `AssertionError` if any hard
criterion is violated. **Do not suppress this error** — a failure means the
downstream pipeline (rasters, training, Mars inference) must not proceed
until the flagged issues are resolved.

In [ ]:
if flags:
    msg = (
        "Stage 5 QA FAILED.\n"
        f"{len(flags)} hard criterion/criteria violated:\n"
        + "\n".join(f"  ✗ {f}" for f in flags)
        + "\n\nDo NOT regenerate rasters or retrain models until these are resolved."
        + "\nSee AGENT_STATE.md for remediation options."
    )
    raise AssertionError(msg)

# ── Soft summary ──────────────────────────────────────────────────────
print("Stage 5 QA PASSED.")
print(f"  Hard flags : 0")
print(f"  Soft warns : {len(warnings)}")
if warnings:
    print("  (warnings do not block the pipeline — review manually)")
    for w in warnings:
        print(f"    ⚠ {w}")

# ── Write QA report ───────────────────────────────────────────────────
if WRITE_REPORT:
    report = summary.copy()
    report['has_warning'] = report['basin'].isin(warned_basins)
    out = RESULTS_DIR / "stage5_earth_network_qa_report.csv"
    report.to_csv(out, index=False)
    print(f"\nQA report written to: {out}")